# Configuración del entorno en Colab

Ejecutar al inicio de **cada sesión** de Colab. Hace lo siguiente:

1. Verifica la GPU asignada.
2. Monta Google Drive.
3. Clona el repositorio (o lo actualiza si ya existe).
4. Instala las dependencias.
5. Descomprime el dataset desde Drive al disco local de Colab (`/content`), en `data/raw/`.

> Se entrena leyendo desde el disco local, no desde Drive: leer miles de imágenes directo desde Drive es muy lento y deja a la GPU esperando.

Se usa el **CCT completo en resolución original** (`cct_images.tar.gz`, 105 GB). Descomprimido ocupa unos 105 GB, así que la máquina de Colab debe tener disco suficiente: la celda 5 lo verifica antes de empezar. La descompresión tarda bastante.

**Estructura esperada en Drive** (ajustar `DRIVE_DATA_DIR` si es otra):
```
MyDrive/redes-parcial-p3/
  cct_images.tar.gz                  imágenes del CCT completo (105 GB)
  annotations/
    caltech_bboxes_20200316.json     bounding boxes
    caltech_camera_traps.json.zip    etiquetas por imagen, fechas, secuencias, cámaras
    CaltechCameraTrapsSplits_v0.json partición por cámara (incluye eccv_train / eccv_val)
    eccv_18_annotations.tar.gz       splits ECCV 2018 (cis / trans)
```

## 1. GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Montar Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Clonar / actualizar el repositorio

In [ ]:
import os

REPO_URL = 'https://github.com/Roxana-Vargas/redes-parcial-p3.git'
REPO_DIR = '/content/redes-parcial-p3'

if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}

## 4. Dependencias

In [ ]:
!pip install -q -r requirements.txt

import ultralytics
ultralytics.checks()

## 5. Descomprimir el dataset en `/content`

In [ ]:
import glob, shutil

DRIVE_DATA_DIR = '/content/drive/MyDrive/redes-parcial-p3'
RAW_DIR = os.path.join(REPO_DIR, 'data', 'raw')
IMAGES_TAR = os.path.join(DRIVE_DATA_DIR, 'cct_images.tar.gz')
DONE_FLAG = os.path.join(RAW_DIR, '.images_ok')
MIN_FREE_GB = 120  # 105 GB de imágenes + margen para labels YOLO y salidas

assert os.path.isfile(IMAGES_TAR), f'No existe {IMAGES_TAR}: revisar la ruta en Drive'
os.makedirs(RAW_DIR, exist_ok=True)

# Anotaciones: archivos pequeños, se copian y se descomprimen
ann_dst = os.path.join(RAW_DIR, 'annotations')
shutil.copytree(os.path.join(DRIVE_DATA_DIR, 'annotations'), ann_dst, dirs_exist_ok=True)
for z in glob.glob(os.path.join(ann_dst, '*.zip')):
    !unzip -oq "{z}" -d "{ann_dst}"
for t in glob.glob(os.path.join(ann_dst, '*.tar.gz')):
    !tar -xzf "{t}" -C "{ann_dst}"

# Imágenes: se descomprimen directo desde Drive (sin copiar antes el .tar.gz)
if os.path.exists(DONE_FLAG):
    print('Imágenes ya descomprimidas en esta sesión.')
else:
    free_gb = shutil.disk_usage('/content').free / 1e9
    print(f'Disco libre en /content: {free_gb:.0f} GB')
    assert free_gb > MIN_FREE_GB, (
        f'Se necesitan más de {MIN_FREE_GB} GB libres. '
        'Cambiar a un entorno con más disco (L4 / A100).')
    !apt-get -qq install -y pv pigz > /dev/null
    !pv -f "{IMAGES_TAR}" | pigz -dc | tar -x -C "{RAW_DIR}"
    open(DONE_FLAG, 'w').close()

!du -sh {RAW_DIR}/*

## 6. Verificación rápida

In [ ]:
n_images = sum(1 for p in glob.glob(os.path.join(RAW_DIR, '**', '*.jpg'), recursive=True))
json_files = glob.glob(os.path.join(RAW_DIR, '**', '*.json'), recursive=True)

print(f'Imágenes .jpg: {n_images:,}')
print('Anotaciones JSON:')
for p in sorted(json_files):
    print('  ', os.path.relpath(p, RAW_DIR))